# Applied Machine Learning, Clustering 3
## Profiling investors with a household survey: segments, wealth, and the biases they hide

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### Where we are coming from

In the first clustering notebook we segmented a bank's customer base: 5,660 rows, eight columns, a clean table. Today
we do the same with a more serious dataset and a more serious consequence. The dataset is the **Survey of Consumer
Finances** of the US Federal Reserve, the best public survey on household finances: net worth, income, debts, which
assets they hold, how much they know about finance and how much risk they say they are willing to take. And the
consequence is that an investor segmentation does not stay in a report: it is used to **assign products**. Every
institution that advises on investments classifies its clients into a profile (conservative, moderate, aggressive);
in Chile the CMF regulation requires it, and the AFP pension multifunds do the same with millions of people.

Two new things compared with the bank. First, a survey is not just any table: it comes with **weights** (each
household represents a different number of households in the country) and **multiple imputations** (each household
appears five times), and if you ignore them the numbers come out wrong by a large factor. Second, and this is the
center of the notebook: the segments the algorithm finds look a lot like the wealth structure of the country, and
that structure is not neutral with respect to sex or race. We will never give the algorithm anyone's sex or race, and
even so the segments will have very different composition by sex and race. We are going to measure how much, and
discuss what it means.

### What you will learn today

1. What an investor profile is and why a segmentation turns into a decision about people.
2. How to work with a complex survey design: implicates and weights, and how wrong you are if you ignore them.
3. How to prepare money variables with very long tails (signed logarithm) and how to segment with them.
4. How to profile segments with weighted statistics, and how to name them.
5. How to measure whether the segments reproduce differences by sex and race, why it happens even though those variables are not used, and what can be done about it.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, os, io, zipfile, urllib.request

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
SEMILLA = 42


## 1. The problem: classifying people, not predicting prices

An investment advisor (an asset manager, a private bank, a pension fund) needs to know which portfolio to recommend
to each client. The standard practice is a profile questionnaire and a rule that converts it into a category. The
temptation to automate is obvious: there are data on thousands of clients, and a clustering algorithm can find
"types" of investor without anyone designing the questionnaire.

The problem is that the types the algorithm finds are types of **financial situation**, and the financial situation
of a country has a history. If the households with the least wealth land in the "conservative" segment and that
segment is offered products with lower expected return, the individual assignment looks reasonable and the aggregate
effect is that access to return becomes conditional on prior wealth. And since prior wealth is correlated with sex and
race for reasons that have nothing to do with risk preferences, the algorithm ends up reproducing that gap without
ever having seen a sex or race column.

None of this is an argument against segmenting. It is an argument for **measuring** the composition of the segments
before using them, which is what we are going to do.

### The data

**Survey of Consumer Finances 2022**, summary extract (the `SCFP2022` file published by the Federal Reserve): 4,595
households, with more than 300 variables. We will use just a few:


In [ ]:

VARIABLES = {
    "AGE": "age of the household head",
    "EDCL": "education: 1 no high school, 2 high school, 3 some college, 4 college degree",
    "KIDS": "number of children in the household",
    "INCOME": "annual household income (USD)",
    "NETWORTH": "net worth (USD; can be negative)",
    "FIN": "financial assets (USD)",
    "FINLIT": "financial literacy: correct answers from 0 to 3",
    "NOFINRISK": "1 if the household declares it is not willing to take any financial risk",
    "HEQUITY": "1 if the household holds stocks (directly or through funds)",
    "HHSEX": "sex of the household head: 1 male, 2 female",
    "RACECL4": "racial group: 1 white, 2 black, 3 hispanic, 4 other",
    "OCCAT1": "occupation of the household head: 1 employee, 2 self-employed, 3 retired or disabled, 4 not working",
    "MARRIED": "1 married or living with a partner, 2 neither married nor living with a partner",
    "WGT": "weight: how many households in the country this row represents",
    "YY1": "household identifier",
    "Y1": "identifier of the household implicate (imputed version)",
}
URL_DATOS = ""      # <-- link to the course file (SCFP2022.csv) if you have it; otherwise it is downloaded from the Federal Reserve (about 20 MB)
URL_FED = "https://www.federalreserve.gov/econres/files/scfp2022excel.zip"

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_DATOS:
    scf = pd.read_csv(normalizar_link(URL_DATOS))
elif os.path.exists("SCFP2022.csv"):
    scf = pd.read_csv("SCFP2022.csv")
else:
    print("Downloading the SCF 2022 from the Federal Reserve...")
    peticion = urllib.request.Request(URL_FED, headers={"User-Agent": "Mozilla/5.0"})     # the server rejects requests without a User-Agent
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(peticion, timeout=300).read())) as z:
        z.extractall(".")
    scf = pd.read_csv("SCFP2022.csv")
scf = scf[list(VARIABLES)]
print("Rows:", len(scf), "  Distinct households:", scf["YY1"].nunique(), "  Rows per household:", scf.groupby("YY1").size().unique())
scf.head()


## 2. Two traps in the survey

### Trap 1: 22,975 rows that are 4,595 households

The SCF publishes **five versions of each household**. Many financial answers come in incomplete (people do not
remember the balance of their retirement account, or do not want to say how much they have), and instead of dropping
those households or filling in with the mean, the Fed generates five sets of plausible values for what is missing:
five **implicates**. The variability among them measures how much we do not know. The demographic variables are the
same in all five; the financial ones vary.

For us this has an immediate consequence: if we segment the 22,975 rows, each household enters five times and the
segments get inflated with copies. We work with **the first implicate of each household** (4,595 rows), and in
exercise 3 we check how different it would come out with another one.


In [ ]:

ejemplo = scf[scf["YY1"] == scf["YY1"].iloc[0]]
print("The five versions of the first household (demographics coincide, financials change):")
print(ejemplo[["YY1", "Y1", "AGE", "EDCL", "INCOME", "NETWORTH", "FIN", "WGT"]].to_string(index=False))
hogares = scf.sort_values(["YY1", "Y1"]).groupby("YY1", as_index=False).first()
print()
print("Households we work with:", len(hogares))


### Trap 2: the weights

The SCF is not a simple random sample: it **oversamples very wealthy households**, because otherwise it would barely
capture the top of the distribution, where most financial assets are. That is why each row comes with a weight
`WGT`, how many households in the country it represents. An unweighted mean describes the sample (full of rich
people), not the population. The difference is not a detail:


In [ ]:

def media_ponderada(x, w):
    return np.average(x, weights=w)

def mediana_ponderada(x, w):
    orden = np.argsort(x)
    x_, w_ = np.asarray(x)[orden], np.asarray(w)[orden]
    return np.interp(0.5, np.cumsum(w_) / np.sum(w_), x_)

filas = []
for v in ["INCOME", "NETWORTH", "FIN"]:
    filas.append(dict(variable=v, media_sin_ponderar=hogares[v].mean(), media_ponderada=media_ponderada(hogares[v], hogares["WGT"]),
                      mediana_sin_ponderar=hogares[v].median(), mediana_ponderada=mediana_ponderada(hogares[v], hogares["WGT"])))
tabla = pd.DataFrame(filas).set_index("variable")
tabla["the unweighted mean is ... times the real one"] = tabla["media_sin_ponderar"] / tabla["media_ponderada"]
print(tabla.round(0).to_string())


The unweighted mean net worth is more than ten times the real one, and the unweighted median is also well above:
the sample is full of wealthy households on purpose. The weighted median net worth, close to 190,000 dollars, is the
figure the Fed reports for 2022. **Rule:** before touching a survey, look for the weight and count the implicates.

For clustering there is a nuance. K-means accepts weights (`sample_weight`) and we will use them so that the
centroids represent the population and not the sample of wealthy people. But the **descriptive statistics** of each
segment (average age, median net worth, percentage of women) must always be computed weighted, and we do that by hand
with the two functions above.

## 3. Preparing the variables

The three money variables have enormous tails (net worth goes from −555,000 to more than 2 billion dollars). With
those tails, k-means would devote all its groups to separating the few households at the very top. The standard
transformation is the **signed logarithm**, $\operatorname{sign}(x)\,\ln(1 + |x|)$, which compresses the tails and
accepts zeros and negatives. Then we scale, as always.

The choice of variables is a decision that must be declared: we use age, education, children, financial literacy
and the three money variables. **We do not use sex or race**, and we leave them out on purpose, so that we can ask
afterwards whether the segments reconstruct them on their own.

The same rule as in the previous notebook applies to how the variables enter. `EDCL` is a 1 to 4 scale defined by the
survey itself, and we use it as such with the equal-steps assumption declared; `KIDS` and `FINLIT` are counts. The
survey's nominal variables (occupation `OCCAT1`, marital status `MARRIED`, housing tenure `HOUSECL`) stay out of the
segmentation, and if we wanted to include them we would do what we did with the bank: define a business axis and
assign scores by hand, never blind one-hot, which would put a married household as far from a separated one as from a
single one and would multiply the dimensions. Exercise 4 does this.


In [ ]:

datos = hogares.copy()
for v in ["INCOME", "NETWORTH", "FIN"]:
    datos["log_" + v] = np.sign(datos[v]) * np.log1p(np.abs(datos[v]))
SEGMENTAR = ["AGE", "EDCL", "KIDS", "FINLIT", "log_INCOME", "log_NETWORTH", "log_FIN"]
Z = StandardScaler().fit_transform(datos[SEGMENTAR])
w = datos["WGT"].values
fig = px.histogram(datos, x="log_NETWORTH", nbins=60, title="Net worth in signed logarithm (negative = debts larger than assets)")
fig.update_layout(height=340, width=720)
fig.show()


## 4. How many segments

The same two criteria as in the previous notebook, with the weight in the fit.


In [ ]:

filas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z, sample_weight=w)
    filas.append(dict(K=k, inercia=km.inertia_, silueta=silhouette_score(Z, km.labels_, sample_size=3000, random_state=SEMILLA)))
criterios = pd.DataFrame(filas).set_index("K")
print(criterios.round(3).to_string())
fig = go.Figure()
fig.add_scatter(x=criterios.index, y=criterios["inercia"], mode="lines+markers", name="inertia")
fig.add_scatter(x=criterios.index, y=criterios["silueta"], mode="lines+markers", name="silhouette", yaxis="y2")
fig.update_layout(title="Inertia and silhouette by number of segments", xaxis_title="K", yaxis=dict(title="inertia"), yaxis2=dict(title="silhouette", overlaying="y", side="right"), height=400, width=720)
fig.show()


Neither does the inertia show an elbow nor the silhouette a clear maximum: households form a **continuum** of
financial situations, not islands, and any $K$ is a cut of that continuum. That is normal with data about people. We
choose **K = 4** because it produces segments that can be described and because it is the number of profiles the
industry usually uses (conservative, moderate, aggressive, plus one of "no saving capacity"). With K = 6, exercise 1
checks whether the conclusions change.


In [ ]:

K = 4
modelo = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z, sample_weight=w)
datos["segmento"] = modelo.labels_

def perfil_segmentos(d, col="segmento"):
    g = d.groupby(col)
    return pd.DataFrame({
        "households in sample": g.size(),
        "% of population": g["WGT"].sum() / d["WGT"].sum(),
        "age": g.apply(lambda x: media_ponderada(x["AGE"], x["WGT"])),
        "education (1 to 4)": g.apply(lambda x: media_ponderada(x["EDCL"], x["WGT"])),
        "children": g.apply(lambda x: media_ponderada(x["KIDS"], x["WGT"])),
        "fin. literacy (0 to 3)": g.apply(lambda x: media_ponderada(x["FINLIT"], x["WGT"])),
        "median income": g.apply(lambda x: mediana_ponderada(x["INCOME"], x["WGT"])),
        "median net worth": g.apply(lambda x: mediana_ponderada(x["NETWORTH"], x["WGT"])),
        "median fin. assets": g.apply(lambda x: mediana_ponderada(x["FIN"], x["WGT"])),
        "% no risk tolerance": g.apply(lambda x: media_ponderada(x["NOFINRISK"], x["WGT"])),
        "% holding stocks": g.apply(lambda x: media_ponderada(x["HEQUITY"], x["WGT"])),
    })

perfil = perfil_segmentos(datos)
print(perfil.round(2).to_string())


## 5. Reading the segments

Four clear profiles, ordered by wealth and stage of life (the segment numbers are arbitrary; the names are ours,
chosen by looking at the table):

- **Consolidated wealth** (32% of the population): 62 years old on average, median net worth of 850,000 dollars and
  380,000 in financial assets, the highest education and financial literacy, 87% holding stocks and only one in five
  declares no risk tolerance.
- **Professionals with family** (34%): 36 years old, 1.4 children, median income of 80,000 dollars, wealth under
  construction (122,000), high education, 62% holding stocks, 28% with no risk tolerance.
- **Low wealth** (27%): 65 years old, median income of 38,000 dollars, barely 4,000 in financial assets (the median
  net worth of 97,000 is almost all the home), the lowest education and literacy, 26% holding stocks and **68% with
  no risk tolerance**.
- **Young indebted** (8%): 41 years old, **negative** net worth, almost no financial assets, 48% with no risk
  tolerance.

Declared risk aversion follows wealth: 21 to 28% in the two segments with money, 48 to 68% in the two without. That
is the result that would make an automated advisor offer conservative products to those who have the least, and it
is the starting point of the next section.


In [ ]:

NOMBRES = perfil["median net worth"].rank().map({4.0: "consolidated wealth", 3.0: "professionals with family", 2.0: "low wealth", 1.0: "young indebted"})
datos["perfil"] = datos["segmento"].map(NOMBRES)
ORDEN = ["consolidated wealth", "professionals with family", "low wealth", "young indebted"]

centroides = pd.DataFrame(modelo.cluster_centers_, columns=SEGMENTAR)
centroides.insert(0, "perfil", [NOMBRES[k] for k in range(K)])
fig = px.parallel_coordinates(centroides.assign(id=range(K)), color="id", dimensions=SEGMENTAR, title="Centroids in standardized units (0 = average; each line is a segment)")
fig.update_layout(height=440)
fig.show()
muestra = datos.sample(2500, random_state=SEMILLA)
fig = px.scatter(muestra, x="AGE", y="log_NETWORTH", color="perfil", category_orders={"perfil": ORDEN}, opacity=0.55,
                 labels={"AGE": "age", "log_NETWORTH": "net worth (signed log)"}, title="The segments in the age and net worth plane (2,500 random households)")
fig.update_layout(height=460, width=820)
fig.show()


In [ ]:

resumen = perfil.copy(); resumen.index = [NOMBRES[k] for k in resumen.index]; resumen = resumen.loc[ORDEN]
fig = go.Figure()
fig.add_bar(x=ORDEN, y=resumen["% no risk tolerance"], name="% no risk tolerance")
fig.add_bar(x=ORDEN, y=resumen["% holding stocks"], name="% holding stocks")
fig.add_hline(y=media_ponderada(datos["NOFINRISK"], datos["WGT"]), line=dict(dash="dot"), annotation_text="national average with no tolerance")
fig.update_layout(barmode="group", title="Risk attitude and risk holding by segment (weighted)", height=400, width=760, yaxis_tickformat=".0%")
fig.show()


## 6. The biases the segments hide

Now the central question. We never used sex or race. Are the segments neutral with respect to them?

Two ways to look at it, and it is worth doing both. The **composition** of each segment (what percentage of its
households has a female head, or is black or hispanic) compared with the population; and the **distribution** of
each group across the segments (of all hispanic households, what fraction falls in "low wealth"). The second is the
one that matters for an assignment rule: if the "consolidated wealth" segment is offered the products with the
highest expected return, that fraction is the **access rate** of each group to those products.


In [ ]:

datos["sexo"] = datos["HHSEX"].map({1: "male", 2: "female"})
datos["raza"] = datos["RACECL4"].map({1: "white", 2: "black", 3: "hispanic", 4: "other"})

def composicion(d, grupo):
    t = d.groupby(["perfil", grupo])["WGT"].sum().unstack(grupo)
    t = t.div(t.sum(axis=1), axis=0).loc[ORDEN]
    t.loc["population"] = d.groupby(grupo)["WGT"].sum() / d["WGT"].sum()
    return t

def distribucion(d, grupo):
    t = d.groupby([grupo, "perfil"])["WGT"].sum().unstack("perfil")[ORDEN]
    return t.div(t.sum(axis=1), axis=0)

print("Composition of each segment by sex of the household head (weighted), against the population:")
print(composicion(datos, "sexo").round(2).to_string())
print()
print("Composition by racial group:")
print(composicion(datos, "raza").round(2).to_string())
print()
print("Distribution of each group across the segments (each row sums to 1):")
dist_sexo, dist_raza = distribucion(datos, "sexo"), distribucion(datos, "raza")
print(pd.concat([dist_sexo, dist_raza]).round(2).to_string())


In [ ]:

largo = pd.concat([dist_sexo, dist_raza]).reset_index().melt(id_vars="index", var_name="perfil", value_name="fraccion").rename(columns={"index": "grupo"})
fig = px.bar(largo, x="perfil", y="fraccion", color="grupo", barmode="group", category_orders={"perfil": ORDEN},
             title="What fraction of each group falls in each segment (weighted)")
fig.update_layout(height=430, width=860, yaxis_tickformat=".0%")
fig.show()


The numbers are large and it is worth saying them plainly. 35% of households with a female head fall in "low wealth"
and 15% in "young indebted", against 23% and 5% of households with a male head; in "consolidated wealth" the
relationship reverses (24% against 35%). Among hispanic households, 36% are in "low wealth" and only 11% in
"consolidated wealth"; among white households, 26% and 38%. And the indebted segment has a majority of female heads
(53%) and one in four families is black, twice their share of the population. If the rule were "products with
higher expected return for the consolidated segment", the access rate of a white household would be **three and a
half times** that of a hispanic household.

The algorithm did not do this: the **wealth structure** of the country did, and the algorithm described it
accurately. The variables we used (net worth, income, education, financial literacy) are **proxies** for sex and
race because in the United States, and in Chile, wealth is correlated with them for historical reasons. Deleting the
protected columns does not delete their effect; it only prevents measuring it. We measure it with a test and an
effect size:


In [ ]:

def cramer_v(tabla):
    chi2 = stats.chi2_contingency(tabla)[0]
    n = tabla.values.sum()
    return np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))

for grupo in ["sexo", "raza"]:
    tabla = pd.crosstab(datos["perfil"], datos[grupo])              # sample counts (the test needs counts, not weights)
    chi2, p, gl, _ = stats.chi2_contingency(tabla)
    print(f"Segment x {grupo}: chi-squared {chi2:.0f}, p-value {p:.1e}, Cramér's V {cramer_v(tabla):.3f}")
print()
print("Cramér's V goes from 0 (independence) to 1 (total determination). Values from 0.1 to 0.3 are moderate")
print("associations: the segments do not determine sex or race, but they are far from independent of them.")


### What to do with this

There are three answers, and all three are policy decisions, not modeling decisions:

1. **Measure and report.** The minimum requirement: every segmentation used to assign products should come with its
   composition table by protected groups. For that you have to **have** the protected variables in the dataset, even
   if they are not used to segment. The intuition of "better not to collect them" produces segmentations that are
   just as biased and impossible to audit.
2. **Separate tolerance from capacity.** The 59% "no risk tolerance" in the low wealth segment mixes two things:
   households that do not want risk and households that cannot absorb losses. A profile questionnaire should ask
   about both separately, and the recommendation should depend on the second as much as on the first.
3. **Define what "fair" means before using the rule.** If the assignment requires equal access rates across groups
   (demographic parity), the rule will have to be intervened; if it requires that the same financial situation gets
   the same recommendation, the current rule already satisfies it and the gap is the country's. Both definitions are
   reasonable and they are incompatible; choosing one is a decision the algorithm cannot make.

## 7. What to take away

1. A survey comes with weights and implicates. Ignoring them inflates the mean net worth more than ten times and counts each household five times.
2. Money variables are segmented in signed logarithm; without that, k-means only separates the richest.
3. With household data there is no correct K: there is a continuum and a useful cut. K = 4 gives profiles the industry recognizes.
4. The segments follow wealth, and so does declared risk aversion: 21 to 28% in the segments with money, 48 to 68% in those without.
5. Without using sex or race, the segments have very different composition by sex and race, because the financial variables are proxies. The access rate to the "consolidated" segment is three and a half times higher for a white household than for a hispanic one.
6. Deleting the protected variables fixes nothing; measuring and reporting does. What to do next is a policy decision.


## Exercises

1. **More segments.** Repeat the segmentation with K = 6, name the segments, and recompute the distribution of each
   racial group across segments. Does the access gap to the highest-wealth segment shrink, grow, or spread out?
2. **No money.** Segment using only age, education, children and financial literacy (no income, net worth or
   assets). Do the segments still have different composition by sex and race? Which variable is left acting as a
   proxy?
3. **Another implicate.** Repeat the segmentation with the second implicate of each household (`Y1` ending in 2) and
   compare with the first one using the adjusted Rand index. How much does a household's assignment change depending
   on which imputed version it got?
4. **Adding categoricals with scores.** Include occupation (`OCCAT1`) and marital status (`MARRIED`) in the
   segmentation, as we did with the bank: define business axes (expected income and stability for occupation,
   household commitment for marital status), declare the scores, segment again with K = 4 and compare with the base
   segmentation (adjusted Rand, profiles) and, above all, recompute Cramér's V with sex and race. Does adding variables
   that are not sex change how much the segments are associated with sex?


## Solutions


In [ ]:

# Exercise 1: K = 6
m6 = KMeans(n_clusters=6, n_init=20, random_state=SEMILLA).fit(Z, sample_weight=w)
d6 = datos.copy(); d6["segmento"] = m6.labels_
p6 = perfil_segmentos(d6)
p6 = p6.sort_values("median net worth", ascending=False)
print(p6[["% of population", "age", "median income", "median net worth", "% no risk tolerance", "% holding stocks"]].round(2).to_string())
d6["perfil"] = d6["segmento"].map({s: f"seg {s} (median net worth {p6.loc[s, 'median net worth']:,.0f})" for s in p6.index})
orden6 = [f"seg {s} (median net worth {p6.loc[s, 'median net worth']:,.0f})" for s in p6.index]
t = d6.groupby(["raza", "perfil"])["WGT"].sum().unstack("perfil")[orden6]
print()
print("Distribution of each racial group across the six segments (ordered from most to least wealth):")
print(t.div(t.sum(axis=1), axis=0).round(2).to_string())


With six segments the gap does not disappear: it spreads out. The highest-wealth segment still concentrates white
households well above their population share, and the two lowest-wealth segments still concentrate black and
hispanic households. Changing $K$ changes the granularity of the description, not the structure it describes.


In [ ]:

# Exercise 2: segmenting without money variables
SIN_DINERO = ["AGE", "EDCL", "KIDS", "FINLIT"]
Z2 = StandardScaler().fit_transform(datos[SIN_DINERO])
m2 = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z2, sample_weight=w)
d2 = datos.copy(); d2["segmento"] = m2.labels_
p2 = perfil_segmentos(d2)
d2["perfil"] = d2["segmento"].map({s: f"seg {s}" for s in p2.index})
print(p2[["% of population", "age", "education (1 to 4)", "children", "fin. literacy (0 to 3)", "median net worth", "% no risk tolerance"]].round(2).to_string())
for grupo in ["sexo", "raza"]:
    tabla = pd.crosstab(d2["perfil"], d2[grupo])
    print(f"Cramér's V segment x {grupo}: {cramer_v(tabla):.3f}   (with money it was {cramer_v(pd.crosstab(datos['perfil'], datos[grupo])):.3f})")
t = d2.groupby(["raza", "perfil"])["WGT"].sum().unstack("perfil")
print()
print(t.div(t.sum(axis=1), axis=0).round(2).to_string())


Without the money variables the association with race goes down but does not disappear, because education and
financial literacy are also correlated with racial group. And the resulting segments are worse for the original
purpose (they no longer separate by wealth, which is what matters for assigning products). Removing variables is
not a solution: it changes which proxy does the work.


In [ ]:

# Exercise 3: the second implicate
segundo = scf.sort_values(["YY1", "Y1"]).groupby("YY1", as_index=False).nth(1).set_index("YY1").loc[hogares["YY1"]].reset_index()
d3 = segundo.copy()
for v in ["INCOME", "NETWORTH", "FIN"]:
    d3["log_" + v] = np.sign(d3[v]) * np.log1p(np.abs(d3[v]))
Z3 = StandardScaler().fit_transform(d3[SEGMENTAR])
etiquetas_3 = modelo.predict(Z3)                      # same centroids, another imputed version of each household
print("Households that change segment between the first and the second implicate:", f"{(etiquetas_3 != modelo.labels_).mean():.1%}")
m3 = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z3, sample_weight=d3["WGT"].values)
print("Adjusted Rand between a segmentation made with implicate 1 and another made with implicate 2:", round(adjusted_rand_score(modelo.labels_, m3.labels_), 3))


The great majority of households stay in the same segment with any of their imputed versions, and the two
segmentations almost coincide: imputation moves the households close to a boundary, not the structure. For a serious
report the right thing is to repeat the analysis with all five implicates and report the range; for the class, the
first one is enough and the check in this exercise is what gives the peace of mind.


In [ ]:

# Exercise 4: occupation and marital status as scores on declared axes (same logic as in the bank notebook)
PUNTAJE_INGRESO_OCUP = {1: 60, 2: 75, 3: 30, 4: 0}       # employee, self-employed, retired, not working
PUNTAJE_ESTAB_OCUP = {1: 80, 2: 40, 3: 100, 4: 0}        # a pension is the most predictable income; not working, the least
PUNTAJE_COMPROMISO = {1: 100, 2: 0}                       # married or living with a partner, against neither
d4 = datos.copy()
d4["ingreso_ocup"] = d4["OCCAT1"].map(PUNTAJE_INGRESO_OCUP)
d4["estab_ocup"] = d4["OCCAT1"].map(PUNTAJE_ESTAB_OCUP)
d4["compromiso"] = d4["MARRIED"].map(PUNTAJE_COMPROMISO)
SEGMENTAR_4 = SEGMENTAR + ["ingreso_ocup", "estab_ocup", "compromiso"]
Z4 = StandardScaler().fit_transform(d4[SEGMENTAR_4])
m4 = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z4, sample_weight=w)
d4["perfil4"] = m4.labels_
print("Adjusted Rand against the base segmentation:", round(adjusted_rand_score(modelo.labels_, m4.labels_), 3))
g = d4.groupby("perfil4")
print(pd.DataFrame({"% of population": g["WGT"].sum() / d4["WGT"].sum(), "age": g.apply(lambda x: media_ponderada(x["AGE"], x["WGT"])),
                    "median net worth": g["NETWORTH"].median(), "occup. income": g.apply(lambda x: media_ponderada(x["ingreso_ocup"], x["WGT"])),
                    "occup. stability": g.apply(lambda x: media_ponderada(x["estab_ocup"], x["WGT"])), "compromiso": g.apply(lambda x: media_ponderada(x["compromiso"], x["WGT"])),
                    "% no risk tolerance": g.apply(lambda x: media_ponderada(x["NOFINRISK"], x["WGT"]))}).round(2).to_string())
print()
for grupo in ["sexo", "raza"]:
    print(f"Cramér's V segment x {grupo}: base {cramer_v(pd.crosstab(datos['perfil'], datos[grupo])):.3f}   with occupation and marital status {cramer_v(pd.crosstab(d4['perfil4'], d4[grupo])):.3f}")
print()
print("Composition by sex of the new segments (weighted):")
t = d4.groupby(["perfil4", "sexo"])["WGT"].sum().unstack("sexo")
print(t.div(t.sum(axis=1), axis=0).round(2).to_string())


With the occupation and marital status scores the segmentation changes substantially (adjusted Rand of 0.25): a
segment of retirees appears (28% of households, 72 years old, stability 99), a small one of households where the head
does not work (4%), and the other two are ordered by whether the household is a couple or not, with net worth
following them. And the result that matters for the topic of this notebook: the association with race goes down a
little (Cramér's V from 0.25 to 0.22), but the association with **sex** goes up (from 0.31 to 0.38), and the
highest-wealth segment ends up with 90% male household heads, because in the survey the "head" of a couple household
is almost always the man and the head of a household without a partner is very often a woman. The commitment axis,
defined with all the business logic in the world, is a proxy for the sex of the household head. It is the same lesson
as in section 5 with another variable: declaring the assumptions of the mapping is necessary, but it does not
guarantee neutrality; it has to be measured afterwards, every time.
